<a href="https://colab.research.google.com/github/weagan/ARC-AGI-3/blob/main/duck_harness_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duck Harness Prototype for ARC-AGI-3
This notebook provides a high-level executable prototype of the **Duck Harness** architecture designed by Tufa Labs for the ARC-AGI-3 environment. It showcases:
1. Multi-platform secret handling (Google Colab vs Kaggle).
2. Dual-modal 2D grid visualization/serialization.
3. A code-as-action Python REPL execution loop.
4. Automatic context window eviction supervision.

### 1. Platform-Agnostic Secret Handling
Detects the current environment (Colab or Kaggle) and loads the required API keys securely without hardcoding.

In [1]:
import os

def load_api_key(secret_name="HF_TOKEN"):
    """Loads a secret securely from Google Colab or Kaggle secret managers."""
    # Check for Google Colab
    try:
        from google.colab import userdata
        print("[Info] Google Colab environment detected. Fetching secret...")
        return userdata.get(secret_name)
    except ImportError:
        pass

    # Check for Kaggle
    try:
        from kaggle_secrets import UserSecretsClient
        print("[Info] Kaggle environment detected. Fetching secret...")
        user_secrets = UserSecretsClient()
        return user_secrets.get_secret(secret_name)
    except ImportError:
        pass

    # Fallback to local environment variables
    print("[Info] Local or fallback environment detected.")
    return os.environ.get(secret_name, "MOCK_API_KEY_FOR_LOCAL_RUNNING")

API_KEY = load_api_key()
print("Secret status: Loaded successfully (masked for privacy).")

[Info] Google Colab environment detected. Fetching secret...
Secret status: Loaded successfully (masked for privacy).


### 2. The Mock Game Environment
A lightweight environment representing an ARC-AGI style grid game where the agent must infer rules and execute optimal spatial choices.

In [2]:
import numpy as np

class ArcGameEnvironment:
    """Simulates a simple 2D ARC-style maze where the goal must be discovered."""
    def __init__(self):
        # 0 = Empty, 1 = Wall, 2 = Player, 3 = Goal
        self.grid = np.array([
            [1, 1, 1, 1, 1],
            [1, 2, 0, 0, 1],
            [1, 1, 1, 0, 1],
            [1, 3, 0, 0, 1],
            [1, 1, 1, 1, 1]
        ])
        self.player_pos = (1, 1)
        self.moves = 0

    def get_state(self):
        return self.grid.copy()

    def step(self, direction):
        """Executes a step: 'UP', 'DOWN', 'LEFT', 'RIGHT'"""
        self.moves += 1
        r, c = self.player_pos
        if direction == 'UP': r -= 1
        elif direction == 'DOWN': r += 1
        elif direction == 'LEFT': c -= 1
        elif direction == 'RIGHT': c += 1

        # Check constraints / collision
        if self.grid[r, c] != 1:
            # Move current position
            self.grid[self.player_pos] = 0
            if self.grid[r, c] == 3:
                print("Goal reached autonomously!")
            self.player_pos = (r, c)
            self.grid[r, c] = 2

        return self.get_state(), self.moves

### 3. Duck Harness Supervisor Architecture
Implements the Python REPL interface, text serialization, and history eviction supervision to prevent context explosion.

In [3]:
import sys
from io import StringIO

class DuckHarness:
    def __init__(self, max_context_steps=3):
        self.max_context_steps = max_context_steps
        self.history = []

    def serialize_grid(self, grid):
        """Converts the numerical 2D array into a structural textual string format."""
        return "\n".join([" ".join(map(str, row)) for row in grid])

    def supervise_context(self):
        """Evicts stale historical execution contexts to prevent attention decay."""
        if len(self.history) > self.max_context_steps:
            print(f"[Harness] Evicting oldest step context. Current size: {len(self.history)}")
            self.history = self.history[-self.max_context_steps:]

    def execute_code_repl(self, code_str, env):
        """Runs code generated by the LLM within a safe local interpreter state."""
        old_stdout = sys.stdout
        redirected_output = StringIO()
        sys.stdout = redirected_output

        local_vars = {"env": env}
        try:
            exec(code_str, {}, local_vars)
            error = None
        except Exception as e:
            error = str(e)

        sys.stdout = old_stdout
        return redirected_output.getvalue(), error

### 4. Running the Code-as-Action Loop Simulation
Simulates the model generating a Python pathfinding script step to interact with the environment through the harness layer.

In [4]:
# Instantiate environment and harness supervisor
env = ArcGameEnvironment()
harness = DuckHarness(max_context_steps=2)

# Initial state capture
current_grid = env.get_state()
serialized_view = harness.serialize_grid(current_grid)
harness.history.append({"state": serialized_view})

print("--- Initial Game View ---")
print(serialized_view)

# Simulated programmatic action sequence output by the model reasoning layer
llm_generated_code = """
# Model inferred rule: 2 must reach 3 avoiding 1
env.step('RIGHT')
env.step('RIGHT')
env.step('DOWN')
env.step('DOWN')
env.step('LEFT')
"""

print("\n--- Executing LLM Code-as-Action via Harness REPL ---")
output, err = harness.execute_code_repl(llm_generated_code, env)

if err:
    print(f"Execution Error: {err}")
else:
    print(output if output else "Code executed smoothly with no output noise.")

final_grid = env.get_state()
print("\n--- Final Game View ---")
print(harness.serialize_grid(final_grid))
harness.history.append({"state": harness.serialize_grid(final_grid)})
harness.supervise_context()

--- Initial Game View ---
1 1 1 1 1
1 2 0 0 1
1 1 1 0 1
1 3 0 0 1
1 1 1 1 1

--- Executing LLM Code-as-Action via Harness REPL ---
Code executed smoothly with no output noise.

--- Final Game View ---
1 1 1 1 1
1 0 0 0 1
1 1 1 0 1
1 3 2 0 1
1 1 1 1 1


In [5]:
import google.colab.userdata as userdata
import json

try:
    # Retrieve the list of all secret keys using the colab private API safely
    from google.colab import _message
    secrets_dict = _message.blocking_request('GetSecrets', request={}, timeout_sec=5)

    if secrets_dict and 'secrets' in secrets_dict:
        keys = [secret['key'] for secret in secrets_dict['secrets']]
        print("--- Found the following secrets keys in your Colab workspace ---")
        for key in keys:
            print(f"🔑 {key}")

        # Suggest matches
        free_suggestions = ['GEMINI_API_KEY', 'GOOGLE_API_KEY', 'HF_TOKEN', 'HUGGINGFACE_CO_API_TOKEN', 'GROQ_API_KEY']
        matching_keys = [k for k in keys if k.upper() in free_suggestions or 'API' in k.upper() or 'KEY' in k.upper() or 'TOKEN' in k.upper()]

        if matching_keys:
            print("\n💡 Potential LLM API keys detected:")
            for mk in matching_keys:
                print(f"   - {mk}")
        else:
            print("\nNo obvious free LLM API keys found. You can get a free Gemini API key from Google AI Studio and add it to your Colab secrets as 'GOOGLE_API_KEY'.")
    else:
        print("No secrets found in this Colab notebook environment.")
except Exception as e:
    print(f"Could not automatically list secret names: {e}")
    print("Please check the '🔑' (Secrets) tab on the left sidebar of your Google Colab window to see your saved keys manually!")

No secrets found in this Colab notebook environment.


### 5. Executing with a Real Live Gemini Call
Now we will fetch the live API key from your secrets, serialize the environment state, prompt `gemini-2.5-flash` to write the correct sequence of `.step()` moves, and evaluate the response directly via the `DuckHarness` REPL.

In [6]:
!pip install -q google-genai

In [7]:
from google import genai
from google.colab import userdata

# Securely load your Gemini API Key
gemini_api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_api_key)

# Initialize a fresh environment and harness
env_live = ArcGameEnvironment()
harness_live = DuckHarness(max_context_steps=2)

# Serialize the starting grid to send as a prompt
initial_grid_str = harness_live.serialize_grid(env_live.get_state())

prompt = f"""
You are an agent navigating a grid environment.

Grid rules:
0 = Empty path
1 = Wall (do not enter)
2 = Current Player position
3 = Goal destination

Here is the current state of the grid:
{initial_grid_str}

Your task: Generate Python code to move the player (2) to the goal (3).
Only use the following allowed method calls on the 'env' object:
- env.step('UP')
- env.step('DOWN')
- env.step('LEFT')
- env.step('RIGHT')

Return ONLY the executable Python code inside a markdown code block. Do not write explanations outside of the code block.
"""

print("--- Sending request to Gemini --- ")
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
)

# Extract the code block from response
raw_text = response.text
import re
code_blocks = re.findall(r'```python\s*(.*?)\s*```', raw_text, re.DOTALL)
live_llm_code = code_blocks[0] if code_blocks else raw_text

print("\n--- Live LLM Generated Code ---")
print(live_llm_code)

print("\n--- Executing Live Code via Harness REPL ---")
output, err = harness_live.execute_code_repl(live_llm_code, env_live)

if err:
    print(f"Execution Error: {err}")
else:
    print(output if output else "Code executed smoothly with no output noise.")

print("\n--- Final Grid State After Live Execution ---")
print(harness_live.serialize_grid(env_live.get_state()))

--- Sending request to Gemini --- 

--- Live LLM Generated Code ---
env.step('RIGHT')
env.step('RIGHT')
env.step('DOWN')
env.step('DOWN')
env.step('LEFT')
env.step('LEFT')

--- Executing Live Code via Harness REPL ---
Goal reached autonomously!


--- Final Grid State After Live Execution ---
1 1 1 1 1
1 0 0 0 1
1 1 1 0 1
1 2 0 0 1
1 1 1 1 1


### 6. Testing Unsupervised Reason Inference (True ARC Style)
Here, we ask Gemini to infer the transformation rules strictly using few-shot paired input/output grids, containing absolutely zero textual descriptions of what the game elements or numerical markers mean.

In [4]:
import numpy as np
import sys
import re
import time
from io import StringIO
from google import genai
from google.colab import userdata
from google.genai.errors import ServerError

# Securely load your Gemini API Key
gemini_api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_api_key)

# Explicitly define classes to avoid NameErrors if notebook state was reset
class ArcGameEnvironment:
    def __init__(self):
        self.grid = np.array([
            [1, 1, 1, 1, 1],
            [1, 2, 0, 0, 1],
            [1, 1, 1, 0, 1],
            [1, 3, 0, 0, 1],
            [1, 1, 1, 1, 1]
        ])
        self.player_pos = (1, 1)
        self.moves = 0

    def get_state(self):
        return self.grid.copy()

    def step(self, direction):
        self.moves += 1
        r, c = self.player_pos
        if direction == 'UP': r -= 1
        elif direction == 'DOWN': r += 1
        elif direction == 'LEFT': c -= 1
        elif direction == 'RIGHT': c += 1
        if self.grid[r, c] != 1:
            self.grid[self.player_pos] = 0
            if self.grid[r, c] == 3:
                print("Goal reached autonomously!")
            self.player_pos = (r, c)
            self.grid[r, c] = 2
        return self.get_state(), self.moves

class DuckHarness:
    def __init__(self, max_context_steps=3):
        self.max_context_steps = max_context_steps
        self.history = []

    def serialize_grid(self, grid):
        return "\n".join([" ".join(map(str, row)) for row in grid])

    def execute_code_repl(self, code_str, env):
        old_stdout = sys.stdout
        redirected_output = StringIO()
        sys.stdout = redirected_output
        local_vars = {"env": env}
        try:
            exec(code_str, {}, local_vars)
            error = None
        except Exception as e:
            error = str(e)
        sys.stdout = old_stdout
        return redirected_output.getvalue(), error

# Initialize environments
temp_env = ArcGameEnvironment()
temp_harness = DuckHarness(max_context_steps=2)
initial_grid_str = temp_harness.serialize_grid(temp_env.get_state())

arc_style_prompt = f"""
You are given input grid states, target grid states, and the executable code sequences that transform the input state into the target state.

=== EXAMPLE 1 ===
INPUT GRID:
1 1 1
1 2 1
1 0 1
1 3 1
1 1 1

CODE ACTIONS:
env.step('DOWN')
env.step('DOWN')

TARGET GRID:
1 1 1
1 0 1
1 0 1
1 2 1
1 1 1

=== EXAMPLE 2 ===
INPUT GRID:
1 1 1 1
1 2 0 1
1 1 3 1
1 1 1 1

CODE ACTIONS:
env.step('RIGHT')
env.step('DOWN')

TARGET GRID:
1 1 1 1
1 0 0 1
1 1 2 1
1 1 1 1

=== TEST CASE ===
INPUT GRID:
{initial_grid_str}

Given the rules demonstrated by Example 1 and Example 2, what CODE ACTIONS should you generate for the TEST CASE input grid?
Return ONLY the executable Python code block containing env.step(...) statements inside a markdown code block. Do not provide any natural language text.
"""

# Attempt calling Gemini API with basic retry backoff
response = None
max_retries = 3
for attempt in range(max_retries):
    try:
        print(f"--- Sending ARC-style prompt to Gemini (Attempt {attempt + 1}/{max_retries}) ---")
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=arc_style_prompt,
        )
        break
    except ServerError as e:
        if attempt < max_retries - 1:
            wait_time = (attempt + 1) * 3
            print(f"Gemini servers busy (503). Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            raise e

# Extract the code block from the response
raw_text = response.text
code_blocks = re.findall(r'```python\s*(.*?)\s*```', raw_text, re.DOTALL)
live_llm_code = code_blocks[0] if code_blocks else raw_text

print("\n--- Gemini Output Code (No Explicit Instructions given) ---")
print(live_llm_code)

# Run it against a fresh test env
env_test = ArcGameEnvironment()
harness_test = DuckHarness(max_context_steps=2)

print("\n--- Executing Inferred Actions ---")
output, err = harness_test.execute_code_repl(live_llm_code, env_test)
if err:
    print(f"Execution Error: {err}")
else:
    print(output if output else "Code executed smoothly!")

print("\n--- Final Grid State after inferred behavior ---")
print(harness_test.serialize_grid(env_test.get_state()))

--- Sending ARC-style prompt to Gemini (Attempt 1/3) ---

--- Gemini Output Code (No Explicit Instructions given) ---
env.step('RIGHT')
env.step('RIGHT')
env.step('DOWN')
env.step('DOWN')
env.step('LEFT')
env.step('LEFT')

--- Executing Inferred Actions ---
Goal reached autonomously!


--- Final Grid State after inferred behavior ---
1 1 1 1 1
1 0 0 0 1
1 1 1 0 1
1 2 0 0 1
1 1 1 1 1


### 7. Stress-Testing with a Complex $9 \times 9$ Maze Grid
To push the limits of Gemini's spatial reasoning without explicit instructions, we will instantiate a much larger $9 \times 9$ maze containing multiple dead ends and winding pathways. We will then ask Gemini to generate the sequence of coordinate steps to navigate from start to finish.

In [5]:
class ComplexArcEnvironment:
    """A larger, more complex maze to test pathfinding limits."""
    def __init__(self):
        # 1 = Wall, 0 = Path, 2 = Player, 3 = Goal
        self.grid = np.array([
            [1, 1, 1, 1, 1, 1, 1, 1, 1],
            [1, 2, 0, 1, 0, 0, 0, 0, 1],
            [1, 1, 0, 1, 0, 1, 1, 0, 1],
            [1, 0, 0, 0, 0, 1, 0, 0, 1],
            [1, 0, 1, 1, 1, 1, 0, 1, 1],
            [1, 0, 1, 0, 0, 0, 0, 0, 1],
            [1, 0, 1, 1, 1, 1, 1, 0, 1],
            [1, 0, 0, 0, 0, 0, 1, 3, 1],
            [1, 1, 1, 1, 1, 1, 1, 1, 1]
        ])
        self.player_pos = (1, 1)
        self.moves = 0

    def get_state(self):
        return self.grid.copy()

    def step(self, direction):
        self.moves += 1
        r, c = self.player_pos
        if direction == 'UP': r -= 1
        elif direction == 'DOWN': r += 1
        elif direction == 'LEFT': c -= 1
        elif direction == 'RIGHT': c += 1

        # Move if it's not a wall
        if self.grid[r, c] != 1:
            self.grid[self.player_pos] = 0
            if self.grid[r, c] == 3:
                print("Goal reached autonomously in a complex maze!")
            self.player_pos = (r, c)
            self.grid[r, c] = 2
        return self.get_state(), self.moves

# Instantiate complex setup
complex_env = ComplexArcEnvironment()
complex_harness = DuckHarness(max_context_steps=2)
complex_initial_grid_str = complex_harness.serialize_grid(complex_env.get_state())

print("--- Complex Initial Game View ---")
print(complex_initial_grid_str)

--- Complex Initial Game View ---
1 1 1 1 1 1 1 1 1
1 2 0 1 0 0 0 0 1
1 1 0 1 0 1 1 0 1
1 0 0 0 0 1 0 0 1
1 0 1 1 1 1 0 1 1
1 0 1 0 0 0 0 0 1
1 0 1 1 1 1 1 0 1
1 0 0 0 0 0 1 3 1
1 1 1 1 1 1 1 1 1


In [6]:
complex_arc_prompt = f"""
You are given input grid states, target grid states, and the executable code sequences that transform the input state into the target state.

=== EXAMPLE 1 ===
INPUT GRID:
1 1 1
1 2 1
1 0 1
1 3 1
1 1 1

CODE ACTIONS:
env.step('DOWN')
env.step('DOWN')

TARGET GRID:
1 1 1
1 0 1
1 0 1
1 2 1
1 1 1

=== EXAMPLE 2 ===
INPUT GRID:
1 1 1 1
1 2 0 1
1 1 3 1
1 1 1 1

CODE ACTIONS:
env.step('RIGHT')
env.step('DOWN')

TARGET GRID:
1 1 1 1
1 0 0 1
1 1 2 1
1 1 1 1

=== TEST CASE ===
INPUT GRID:
{complex_initial_grid_str}

Given the rules demonstrated by Example 1 and Example 2, what CODE ACTIONS should you generate for the TEST CASE input grid to transition from state '2' to state '3'?
Return ONLY the executable Python code block containing env.step(...) statements inside a markdown code block. Do not provide any natural language text.
"""

# Call Gemini for the complex test
try:
    print("--- Sending Complex ARC-style prompt to Gemini ---")
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=complex_arc_prompt,
    )

    raw_text = response.text
    code_blocks = re.findall(r'```python\s*(.*?)\s*```', raw_text, re.DOTALL)
    complex_llm_code = code_blocks[0] if code_blocks else raw_text

    print("\n--- Gemini Output Code for Complex Maze ---")
    print(complex_llm_code)

    print("\n--- Executing Inferred Actions ---")
    output, err = complex_harness.execute_code_repl(complex_llm_code, complex_env)
    if err:
        print(f"Execution Error: {err}")
    else:
        print(output if output else "Code executed smoothly!")

    print("\n--- Final Grid State after inferred behavior ---")
    print(complex_harness.serialize_grid(complex_env.get_state()))

except Exception as e:
    print(f"An error occurred: {e}")

--- Sending Complex ARC-style prompt to Gemini ---

--- Gemini Output Code for Complex Maze ---
env.step('RIGHT')
env.step('DOWN')
env.step('DOWN')
env.step('RIGHT')
env.step('RIGHT')
env.step('UP')
env.step('UP')
env.step('RIGHT')
env.step('RIGHT')
env.step('RIGHT')
env.step('RIGHT')
env.step('DOWN')
env.step('DOWN')
env.step('DOWN')
env.step('DOWN')
env.step('DOWN')
env.step('DOWN')

--- Executing Inferred Actions ---
Code executed smoothly!

--- Final Grid State after inferred behavior ---
1 1 1 1 1 1 1 1 1
1 0 0 1 0 0 0 0 1
1 1 0 1 0 1 1 0 1
1 0 0 0 0 1 0 2 1
1 0 1 1 1 1 0 1 1
1 0 1 0 0 0 0 0 1
1 0 1 1 1 1 1 0 1
1 0 0 0 0 0 1 3 1
1 1 1 1 1 1 1 1 1
